# SPARQL-Relax Tutorial

This notebook demonstrates how to use the `sparql-relax` Python bindings to diagnose and repair broken SPARQL queries.

## What is SPARQL-Relax?
SPARQL-Relax helps find why a SPARQL query returns no results when it's expected to, and attempts to find a "connected" version of the query that does return results by searching for paths in the knowledge graph.

In [1]:
from sparql_relax import diagnose, diagnose_and_connect, query, Store
import os

## 1. Setup Data

First, we load our knowledge graph (TTL file). We'll use the `b59` building dataset.

In [2]:
data_path = "/home/lazlo/Desktop/semantics/sparql-relax/eval/buildings/b59.ttl"
with open(data_path, "r") as f:
    data = f.read()
print(f"Loaded data from {data_path}")

Loaded data from /home/lazlo/Desktop/semantics/sparql-relax/eval/buildings/b59.ttl


## 2. Running a Correct Query

Let's start with a query that we know works.

In [3]:
correct_query = """
PREFIX s223: <http://data.ashrae.org/standard223#>
PREFIX ex1: <http://data.ashrae.org/standard223/data/lbnl-example-2#>
PREFIX qudt: <http://qudt.org/schema/qudt/>
PREFIX ref: <https://brickschema.org/schema/Brick/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX quantitykind: <http://qudt.org/vocab/quantitykind/>

SELECT DISTINCT ?target ?label ?alc_endpoint ?mpc_endpoint WHERE {
{ ?target a s223:Zone; s223:hasProperty ?prop. }
UNION
{ ?target a s223:Zone; s223:hasDomainSpace ?space .
?space a ex1:RoomDomainSpace ;
s223:connected/s223:connected/s223:contains/s223:hasProperty ?prop . }

?prop a s223:Property;
qudt:hasQuantityKind|s223:hasEnumerationKind ?quantKind;
rdfs:label ?label;
s223:hasExternalReference ?ref .

?ref ref:endpoint ?alc_endpoint;
ref:mpc_db_name ?mpc_endpoint;
ref:writable "False" .
}
"""

results = query(data, correct_query)
print(f"Found {len(results.bindings)} results.")
for row in results.bindings[:5]:
    print(row)


Found 354 results.
{'target': Term(kind='uri', value='http://data.ashrae.org/standard223/data/lbnl-example-2#04500', datatype=None, language=None), 'label': Term(kind='literal', value='core_zone_remote_air_temp', datatype='http://www.w3.org/2001/XMLSchema#string', language=None), 'alc_endpoint': Term(kind='literal', value='#lbnl_59-bl-072/remote_zone_temp', datatype='http://www.w3.org/2001/XMLSchema#string', language=None), 'mpc_endpoint': Term(kind='literal', value='zone_072c_temp', datatype='http://www.w3.org/2001/XMLSchema#string', language=None)}
{'target': Term(kind='uri', value='http://data.ashrae.org/standard223/data/lbnl-example-2#04500', datatype=None, language=None), 'label': Term(kind='literal', value='zone_co2', datatype='http://www.w3.org/2001/XMLSchema#string', language=None), 'alc_endpoint': Term(kind='literal', value='#lbnl_59-bl-072/zone_co2', datatype='http://www.w3.org/2001/XMLSchema#string', language=None), 'mpc_endpoint': Term(kind='literal', value='zone_072_co2', 

## 3. Diagnosing a Broken Query

Now, let's take a query that is slightly broken (e.g., uses a wrong class). This query should return no results.

In [4]:
broken_query = """
PREFIX s223: <http://data.ashrae.org/standard223#>
PREFIX ex1: <http://data.ashrae.org/standard223/data/lbnl-example-2#>
PREFIX qudt: <http://qudt.org/schema/qudt/>
PREFIX ref: <https://brickschema.org/schema/Brick/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX quantitykind: <http://qudt.org/vocab/quantitykind/>

SELECT DISTINCT ?target ?label ?alc_endpoint ?mpc_endpoint WHERE {
{ ?target a s223:Zone; s223:hasProperty ?prop. }
UNION
{ ?target a s223:Zone; s223:hasDomainSpace ?space .
?space a ex1:RoomDomainSpace ;
s223:connected/s223:connected/s223:contains/s223:hasProperty ?prop . }

?prop a s223:Point; # ERROR ADDED HERE
qudt:hasQuantityKind|s223:hasEnumerationKind ?quantKind;
rdfs:label ?label;
s223:hasExternalReference ?ref .

?ref ref:endpoint ?alc_endpoint;
ref:mpc_db_name ?mpc_endpoint;
ref:writable "False" .
}
"""

results = query(data, broken_query)
print(f"Found {len(results.bindings)} results (expected 0).")

print("Diagnosing...")
diagnosis = diagnose(data, broken_query)
for culprit in diagnosis.culprits:
    print(f"Culprit found: {culprit}")

Found 0 results (expected 0).
Diagnosing...
Culprit found: Culprit(triples=['?prop <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://data.ashrae.org/standard223#Point>'], depth=1)


## 4. Connecting the Query

Now we use `diagnose_and_connect` to automatically find a fix for the broken query.

In [5]:
print("Connecting query...")
report = diagnose_and_connect(data, broken_query)

for result in report.results:
    if result.fixed:
        print(f"Fixed triple: {result.triple} -> {result.path_text}")
        print("Connected Query:\n", result.connected_query)
        
        # Verify the fixed query
        fixed_results = query(data, result.connected_query)
        print(f"Fixed query found {len(fixed_results.bindings)} results.\n")

Connecting query...


### Ignoring cartesian risk

Some broken combinations are skipped by default because checking them could force a costly cross-product evaluation. Pass `ignore_cartesian_risk=True` to force the check anyway.

In [6]:
incorrect_path_query = """
PREFIX s223: <http://data.ashrae.org/standard223#>
PREFIX ex1: <http://data.ashrae.org/standard223/data/lbnl-example-2#>
PREFIX qudt: <http://qudt.org/schema/qudt/>
PREFIX ref: <https://brickschema.org/schema/Brick/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX quantitykind: <http://qudt.org/vocab/quantitykind/>

SELECT DISTINCT ?target ?label ?alc_endpoint ?mpc_endpoint WHERE {
{ ?target a s223:Zone; s223:hasProperty ?prop. }
UNION
{ ?target a s223:Zone; s223:hasDomainSpace ?space .
?space a ex1:RoomDomainSpace ;
s223:connected/s223:connected/s223:contains/s223:hasProperty ?prop . }

?prop a s223:Property;
qudt:hasQuantityKind|s223:hasEnumerationKind ?quantKind;
rdfs:label ?label;
ref:hasExternalReference ?ref . # INCORRECT NAMESPACE on ref:hasExternalReference

?ref ref:endpoint ?alc_endpoint;
ref:mpc_db_name ?mpc_endpoint;
ref:writable "False" .
}
"""

results = query(data, incorrect_path_query)
print(f"Found {len(results.bindings)} results (expected 0).")

print("Connecting...")
diagnosis = diagnose_and_connect(data, incorrect_path_query, ignore_cartesian_risk=True,max_depth = 3)
print(diagnosis)

Found 0 results (expected 0).
Connecting...
ConnectReport(original_row_count=0, results=[ConnectedCulprit(found_at_depth=1, triples=[ConnectedTriple(triple='?prop <https://brickschema.org/schema/Brick/hasExternalReference> ?ref', path_text='<http://data.ashrae.org/standard223#hasExternalReference>')], connected_query='SELECT DISTINCT ?target ?label ?alc_endpoint ?mpc_endpoint WHERE { { ?target <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://data.ashrae.org/standard223#Zone> .?target <http://data.ashrae.org/standard223#hasProperty> ?prop . } UNION { ?target <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://data.ashrae.org/standard223#Zone> .?target <http://data.ashrae.org/standard223#hasDomainSpace> ?space .?space <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://data.ashrae.org/standard223/data/lbnl-example-2#RoomDomainSpace> .?space <http://data.ashrae.org/standard223#connected> _:ab167d4c93c0a1a23bb671f97cb163bd ._:ab167d4c93c0a1a23bb671f97cb163bd <http://da

In [7]:
import re

def pretty_print_sparql(query: str, indent: str = "    ") -> str:
    """
    Formats a compressed SPARQL query into a clean, human-readable structure.
    
    :param query: Raw SPARQL query string.
    :param indent: Indentation string (defaults to 4 spaces).
    :return: Formatted SPARQL query string.
    """
    # 1. Pre-process: ensure full HTTP(S) URIs not enclosed in <> get wrapped
    query = re.sub(r'(?<!<)(https?://[^\s{}()>]+)(?!>)', r'<\1>', query)

    # 2. Tokenize the SPARQL query without breaking URIs or string literals
    pattern = re.compile(r'''
        (?P<IRI_ANGLE><[^>\s]+>)                                 | # <http://...>
        (?P<STRING_DBL>"(?:\\.|[^"\\])*")                        | # "string"
        (?P<STRING_SGL>'(?:\\.|[^'\\])*')                        | # 'string'
        (?P<PAREN_EXPR>\([^)]+\))                                 | # (expression)
        (?P<VAR>\?[a-zA-Z0-9_]+)                                 | # ?variable
        (?P<PREFIXED>:[a-zA-Z0-9_]+|[a-zA-Z0-9_]+:[a-zA-Z0-9_]+) | # :term or prefix:term
        (?P<LBRACE>\{)                                            | # {
        (?P<RBRACE>\})                                            | # }
        (?P<DOT>\.\s|\.$|\.(?=[?\::{}<"]))                       | # Triple terminator dot
        (?P<WORD>[^\s{}()]+)                                      # Other tokens/keywords
    ''', re.VERBOSE)

    raw_tokens = []
    for match in pattern.finditer(query):
        kind = match.lastgroup
        value = match.group().strip()
        if value:
            raw_tokens.append((kind, value))

    lines = []
    current_line = []
    indent_level = 0

    def flush_line():
        nonlocal current_line
        if current_line:
            lines.append(indent * indent_level + " ".join(current_line))
            current_line = []

    i = 0
    while i < len(raw_tokens):
        kind, value = raw_tokens[i]

        # Top-level query verbs
        if kind == 'WORD' and value.upper() in {'SELECT', 'CONSTRUCT', 'ASK', 'DESCRIBE', 'PREFIX', 'BASE'}:
            flush_line()
            current_line.append(value)

        # WHERE clause header
        elif kind == 'WORD' and value.upper() == 'WHERE':
            flush_line()
            if i + 1 < len(raw_tokens) and raw_tokens[i+1][0] == 'LBRACE':
                lines.append(indent * indent_level + "WHERE {")
                indent_level += 1
                i += 1  # Skip the upcoming '{'
            else:
                current_line.append(value)

        # Bottom query modifiers
        elif kind == 'WORD' and value.upper() in {'LIMIT', 'OFFSET', 'ORDER', 'GROUP', 'HAVING', 'VALUES'}:
            flush_line()
            current_line.append(value)

        # UNION keyword
        elif kind == 'WORD' and value.upper() == 'UNION':
            flush_line()
            lines.append(indent * indent_level + "UNION")

        # Opening block brace
        elif kind == 'LBRACE':
            if current_line:
                current_line.append('{')
                flush_line()
            else:
                lines.append(indent * indent_level + "{")
            indent_level += 1

        # Closing block brace
        elif kind == 'RBRACE':
            flush_line()
            indent_level = max(0, indent_level - 1)
            lines.append(indent * indent_level + "}")

        # Statement terminator dot
        elif kind == 'DOT':
            current_line.append('.')
            flush_line()

        else:
            current_line.append(value)

        i += 1

    flush_line()
    return "\n".join(lines)

In [8]:
diagnosis.results[0].triples

[ConnectedTriple(triple='?prop <https://brickschema.org/schema/Brick/hasExternalReference> ?ref', path_text='<http://data.ashrae.org/standard223#hasExternalReference>')]

In [9]:
print(pretty_print_sparql(diagnosis.results[0].pruned_query))

SELECT DISTINCT ?target ?label ?alc_endpoint ?mpc_endpoint
WHERE {
    {
        ?target <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://data.ashrae.org/standard223#Zone> .
        ?target <http://data.ashrae.org/standard223#hasProperty> ?prop .
    }
    UNION
    {
        ?target <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://data.ashrae.org/standard223#Zone> .
        ?target <http://data.ashrae.org/standard223#hasDomainSpace> ?space .
        ?space <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://data.ashrae.org/standard223/data/lbnl-example-2#RoomDomainSpace> .
        ?space <http://data.ashrae.org/standard223#connected> _:ab167d4c93c0a1a23bb671f97cb163bd ._:ab167d4c93c0a1a23bb671f97cb163bd <http://data.ashrae.org/standard223#connected> _:be19a17887176bc9f239b15f33899ed0 ._:be19a17887176bc9f239b15f33899ed0 <http://data.ashrae.org/standard223#contains> _:b2cba3e6d0eeba2960c3ffbbe2a08d29 ._:b2cba3e6d0eeba2960c3ffbbe2a08d29 <http://data.ashrae.org/sta

In [10]:
print(pretty_print_sparql(diagnosis.results[0].connected_query))

SELECT DISTINCT ?target ?label ?alc_endpoint ?mpc_endpoint
WHERE {
    {
        ?target <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://data.ashrae.org/standard223#Zone> .
        ?target <http://data.ashrae.org/standard223#hasProperty> ?prop .
    }
    UNION
    {
        ?target <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://data.ashrae.org/standard223#Zone> .
        ?target <http://data.ashrae.org/standard223#hasDomainSpace> ?space .
        ?space <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://data.ashrae.org/standard223/data/lbnl-example-2#RoomDomainSpace> .
        ?space <http://data.ashrae.org/standard223#connected> _:ab167d4c93c0a1a23bb671f97cb163bd ._:ab167d4c93c0a1a23bb671f97cb163bd <http://data.ashrae.org/standard223#connected> _:be19a17887176bc9f239b15f33899ed0 ._:be19a17887176bc9f239b15f33899ed0 <http://data.ashrae.org/standard223#contains> _:b2cba3e6d0eeba2960c3ffbbe2a08d29 ._:b2cba3e6d0eeba2960c3ffbbe2a08d29 <http://data.ashrae.org/sta

## 5. Efficient Usage with Store

If you are running multiple queries against the same data, use a `Store` to avoid reparsing the TTL file every time.

In [11]:
store = Store(data)
results = store.query(correct_query)
print(f"Store query found {len(results.bindings)} results.")

Store query found 354 results.
